# football-tracking-yolo11 — YOLO11s Futbol Fine-Tune (Google Colab T4)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/turkensar/football-tracking-yolo11/blob/main/notebooks/finetune_colab.ipynb)

Bu not defteri, **football-tracking-yolo11** projesi için `yolo11s.pt` modelini açık lisanslı futbol veri seti üzerinde ince ayar (fine-tune) etmek üzere hazırlanmıştır.

- **Veri seti:** [Roboflow Universe — Football Players Detection (`roboflow-jvuqo/football-players-detection-3zvbc`)](https://universe.roboflow.com/roboflow-jvuqo/football-players-detection-3zvbc) (v1)
- **Lisans:** [CC BY 4.0](https://creativecommons.org/licenses/by/4.0/)
- **Sınıflar (4):** `0: ball`, `1: goalkeeper`, `2: player`, `3: referee`
- **Veri sızıntısı notu:** Yereldeki T11 değerlendirme klibi (2022 Dünya Kupası Ekvador–Katar) bu eğitim setinde yer almaz.

### Çalıştırmadan Önce
1. **Çalışma zamanı (Runtime):** `Çalışma zamanı -> Çalışma zamanı türünü değiştir -> T4 GPU` seçin.
2. **Roboflow API anahtarı (Colab Secrets):** Sol menüdeki **Anahtar (Secrets)** simgesine tıklayın, `ROBOFLOW_API_KEY` adıyla yeni bir gizli anahtar ekleyin ve **Not defteri erişimi** anahtarını açın. Anahtarı asla kod hücrelerine yapıştırmayın.

## 1. GPU Kontrolü

In [ ]:
import subprocess

import torch

subprocess.run(["nvidia-smi"], check=False)

if not torch.cuda.is_available():
    raise RuntimeError(
        "CUDA GPU bulunamadı. Colab menüsünden 'Çalışma zamanı -> "
        "Çalışma zamanı türünü değiştir -> T4 GPU' seçin."
    )

gpu_name = torch.cuda.get_device_name(0)
vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024**3)
print(f"Aktif GPU: {gpu_name} ({vram_gb:.1f} GB VRAM)")

## 2. Bağımlılıkların Kurulumu

In [ ]:
%pip install -q "ultralytics==8.4.216" roboflow

import ultralytics

print("ultralytics:", ultralytics.__version__)

## 3. Veri Setini İndirme (Colab Secrets)

In [ ]:
import os
from pathlib import Path

import yaml
from roboflow import Roboflow

ROBOFLOW_WORKSPACE = "roboflow-jvuqo"
ROBOFLOW_PROJECT = "football-players-detection-3zvbc"
ROBOFLOW_VERSION = 1

api_key = None
try:
    from google.colab import userdata

    api_key = userdata.get("ROBOFLOW_API_KEY")
except Exception:
    api_key = os.environ.get("ROBOFLOW_API_KEY")

if not api_key:
    raise RuntimeError(
        "ROBOFLOW_API_KEY bulunamadı. Colab sol menüsündeki Secrets (Anahtar) bölümünden "
        "'ROBOFLOW_API_KEY' ekleyin ve bu not defterine erişim izni verin."
    )

rf = Roboflow(api_key=api_key)
project = rf.workspace(ROBOFLOW_WORKSPACE).project(ROBOFLOW_PROJECT)
dataset = project.version(ROBOFLOW_VERSION).download("yolov11")

dataset_dir = Path(dataset.location).resolve()
DATA_YAML = dataset_dir / "data.yaml"

# Roboflow bazen gorece '../train/images' yazar; Ultralytics icin yolu kok dizine sabitleyelim.
data_cfg = yaml.safe_load(DATA_YAML.read_text(encoding="utf-8"))
data_cfg["path"] = str(dataset_dir)
for split_key, folder in (
    ("train", "train/images"),
    ("val", "valid/images"),
    ("test", "test/images"),
):
    if (dataset_dir / folder).is_dir():
        data_cfg[split_key] = folder
DATA_YAML.write_text(yaml.safe_dump(data_cfg, sort_keys=False), encoding="utf-8")

print(f"data.yaml konumu: {DATA_YAML}")
print(DATA_YAML.read_text(encoding="utf-8"))

## 4. YOLO11s Fine-Tune (`imgsz=1280`)

In [ ]:
from pathlib import Path

from ultralytics import YOLO

BASE_WEIGHTS = "yolo11s.pt"
EPOCHS = 50
IMGSZ = 1280
BATCH = 8  # T4 (16 GB VRAM) icin 8 veya AutoBatch icin -1; bellek hatasi olursa 4 kullanin.
PATIENCE = 15
SEED = 42
PROJECT = "runs/train"
RUN_NAME = "football_yolo11s"

model = YOLO(BASE_WEIGHTS)
train_results = model.train(
    data=str(DATA_YAML),
    epochs=EPOCHS,
    imgsz=IMGSZ,
    batch=BATCH,
    device=0,
    project=PROJECT,
    name=RUN_NAME,
    patience=PATIENCE,
    seed=SEED,
)

RUN_DIR = Path(model.trainer.save_dir)
BEST_PT = RUN_DIR / "weights" / "best.pt"
RESULTS_CSV = RUN_DIR / "results.csv"
print("Eğitim tamamlandı:", RUN_DIR)
print("En iyi ağırlık:", BEST_PT)

## 5. Doğrulama (`val` + `test` Splitleri: `mAP50`, `mAP50-95` ve `ball`)

In [ ]:
import json

import yaml
from ultralytics import YOLO

best_model = YOLO(str(BEST_PT))
data_cfg = yaml.safe_load(DATA_YAML.read_text(encoding="utf-8"))


def evaluate_split(split_name: str) -> dict:
    metrics = best_model.val(data=str(DATA_YAML), imgsz=IMGSZ, split=split_name)
    result = {
        "overall": {
            "mAP50": round(float(metrics.box.map50), 4),
            "mAP50_95": round(float(metrics.box.map), 4),
            "precision": round(float(metrics.box.mp), 4),
            "recall": round(float(metrics.box.mr), 4),
        },
        "per_class": {},
    }
    for idx, cls_id in enumerate(metrics.box.ap_class_index):
        cls_name = best_model.names[int(cls_id)]
        result["per_class"][cls_name] = {
            "class_id": int(cls_id),
            "AP50": round(float(metrics.box.ap50[idx]), 4),
            "AP50_95": round(float(metrics.box.ap[idx]), 4),
        }
    return result


summary: dict[str, dict | None] = {"val": evaluate_split("val"), "test": None}
if data_cfg.get("test"):
    summary["test"] = evaluate_split("test")
else:
    print("Uyarı: data.yaml içinde 'test' spliti tanımlı değil; yalnızca 'val' değerlendirildi.")

VAL_METRICS_JSON = RUN_DIR / "val_metrics.json"
VAL_METRICS_JSON.write_text(json.dumps(summary, indent=2), encoding="utf-8")

for split_name in ("val", "test"):
    split_metrics = summary.get(split_name)
    if split_metrics is None:
        continue
    print(f"=== {split_name.upper()} Genel Doğrulama Metrikleri ===")
    print(json.dumps(split_metrics["overall"], indent=2))
    print(f"--- {split_name.upper()} Sınıf Bazlı Metrikler ---")
    for cls_name, vals in split_metrics["per_class"].items():
        marker = "  <-- hedef sınıf" if cls_name == "ball" else ""
        print(
            f"  [{vals['class_id']}] {cls_name:<12} "
            f"AP50={vals['AP50']:.4f}  AP50-95={vals['AP50_95']:.4f}{marker}"
        )
    print()

## 6. Çıktıları İndirme (`football_yolo11s.pt`, `results.csv`, `val_metrics.json`)

In [ ]:
import shutil
from pathlib import Path

EXPORT_WEIGHTS = RUN_DIR / "football_yolo11s.pt"
shutil.copy2(BEST_PT, EXPORT_WEIGHTS)
size_mib = EXPORT_WEIGHTS.stat().st_size / (1024**2)
print(f"İndirmeye hazır ağırlık: {EXPORT_WEIGHTS} ({size_mib:.2f} MiB)")

# Opsiyonel: Google Drive'a yedeklemek isterseniz asagidaki satirlari acin.
# from google.colab import drive
# drive.mount("/content/drive")
# drive_out = Path("/content/drive/MyDrive/futbol-cv-runs") / RUN_NAME
# drive_out.mkdir(parents=True, exist_ok=True)
# for src in (EXPORT_WEIGHTS, RESULTS_CSV, VAL_METRICS_JSON):
#     if src.is_file():
#         shutil.copy2(src, drive_out / src.name)
# print("Google Drive yedegi tamamlandi:", drive_out)

try:
    from google.colab import files

    for artifact in (EXPORT_WEIGHTS, RESULTS_CSV, VAL_METRICS_JSON):
        if artifact.is_file():
            files.download(str(artifact))
except ImportError:
    print("Colab disi ortam: dosyalar yerel diskte hazir ->", RUN_DIR)

## Sonraki Adım (Yerel Repo — T13c)

1. İndirilen `football_yolo11s.pt` dosyasını yerel repoda `models/football_yolo11s.pt` konumuna koyun (`models/` `.gitignore` içindedir, git'e eklenmez).
2. `val_metrics.json` (veya `results.csv`) değerlerini saklayın; T13c adımında COCO vs. fine-tune karşılaştırmasına ve `README.md` tablosuna eklenecektir.